# Structure Data Manager

Điều phối ingestion **structure_data** (vnstock → Bronze).

**Chỉ sửa cell `## Cấu hình`** — 4 biến chính:

| Biến | Giá trị | Ý nghĩa |
|------|---------|---------|
| `NOTEBOOK_PROFILE` | `backfill` \| `daily` \| `demo` | OHLCV full / incremental / test 50 mã |
| `UNIVERSE_MODE` | `full_listing` \| `demo_50` | Universe price/company |
| `PRICE_BOARD_MODE` | `full_listing` \| `watchlist_50` | Bảng giá full HOSE/HNX hoặc 50 mã cố định |
| `RUN_ONE_SHOT_PIPELINE` | `True` / `False` | Một lệnh pipeline hoặc từng ô |

Bronze: `data-lake/raw/Structure_Data/` · Docs: `Docs/Structure_data_flow.md`


In [1]:
import os, sys, warnings, threading
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
for stream in (sys.stdout, sys.stderr):
    if hasattr(stream, "reconfigure"):
        stream.reconfigure(encoding="utf-8", errors="replace")
_hook = threading.excepthook
def _ignore_unicode(args):
    if "UnicodeDecodeError" not in str(args.exc_type):
        _hook(args)
threading.excepthook = _ignore_unicode
warnings.filterwarnings("ignore")
print("[OK] UTF-8 guard")


[OK] UTF-8 guard


## Cấu hình

Sửa 4 biến đầu cell dưới rồi chạy.

In [2]:
import importlib
from datetime import datetime

import structure_data as sd
import structure_data.common as sd_common
import structure_data.config as sd_config
import structure_data.price_ingestor as sd_price_ingestor
import structure_data.index_ingestor as sd_index_ingestor
import structure_data.stock_info_ingestor as sd_stock_info_ingestor
import structure_data.pipeline as sd_pipeline

for _m in (sd_common, sd_config, sd_price_ingestor, sd_index_ingestor, sd_stock_info_ingestor, sd_pipeline):
    importlib.reload(_m)
sd = importlib.reload(sd)

from structure_data import (
    DEFAULT_PRICE_BOARD_TICKERS,
    IngestionConfig,
    configure_logging,
    register_vnstock_api_key_from_env,
    run_structure_ingestion_pipeline,
    run_structure_full_ingestion_pipeline,
    ingest_indices,
    ingest_listing,
    ingest_company_overview,
    ingest_financial_ratio,
)
from structure_data.common import load_tickers_from_listing_bronze, listing_bronze_path
from structure_data.pipeline import _ingest_price_batched, _ingest_price_board_snapshot

register_vnstock_api_key_from_env("VNSTOCK_API_KEY")
configure_logging()

# ========== CẤU HÌNH (sửa ở đây) ==========
NOTEBOOK_PROFILE = "backfill"        # backfill | daily | demo
UNIVERSE_MODE = "full_listing"       # full_listing | demo_50
PRICE_BOARD_MODE = "watchlist_50"    # full_listing | watchlist_50 (backfill: 50 mã tại thời điểm chạy)
RUN_ONE_SHOT_PIPELINE = True
INCLUDE_FINANCIAL_RATIO = True

NOTEBOOK_PRESETS = {
    "backfill": {"use_incremental_window": False, "incremental_window_days": 10, "years_back": 5},
    "daily": {"use_incremental_window": True, "incremental_window_days": 1, "years_back": 5},
    "demo": {"use_incremental_window": False, "incremental_window_days": 1, "years_back": 1},
}
UNIVERSE_PRESETS = {
    "demo_50": {"use_listing_as_universe": False, "listing_exchange_filter": [], "listing_security_type_filter": []},
    "full_listing": {"use_listing_as_universe": True, "listing_exchange_filter": ["HOSE", "HNX"], "listing_security_type_filter": ["stock"]},
}
PRICE_BOARD_PRESETS = {
    "full_listing": {"price_board_use_listing_universe": True, "price_board_max_tickers": 5000, "price_board_batched": True},
    "watchlist_50": {"price_board_use_listing_universe": False, "price_board_max_tickers": 50, "price_board_batched": False, "price_board_tickers": list(DEFAULT_PRICE_BOARD_TICKERS)},
}

if NOTEBOOK_PROFILE == "demo":
    UNIVERSE_MODE = "demo_50"
    PRICE_BOARD_MODE = "watchlist_50"

for name, presets, value in (
    ("NOTEBOOK_PROFILE", NOTEBOOK_PRESETS, NOTEBOOK_PROFILE),
    ("UNIVERSE_MODE", UNIVERSE_PRESETS, UNIVERSE_MODE),
    ("PRICE_BOARD_MODE", PRICE_BOARD_PRESETS, PRICE_BOARD_MODE),
):
    if value not in presets:
        raise ValueError(f"{name}={value!r} — chọn {list(presets)}")

cfg = IngestionConfig(
    **UNIVERSE_PRESETS[UNIVERSE_MODE],
    **PRICE_BOARD_PRESETS[PRICE_BOARD_MODE],
    **NOTEBOOK_PRESETS[NOTEBOOK_PROFILE],
    price_batch_size=100,
    price_board_batch_size=50,
    financial_ratio_exchange_filter=["HOSE", "HNX"],
    delay_between_batches_sec=5.0,
    delay_between_categories_sec=30,
    rate_limit_rpm=50,
    inter_request_delay_sec=0.5,
)
cfg.run_partition = datetime.now().strftime("%Y-%m-%dT%H%M%S")
cfg.index_tickers = ["VNINDEX", "VN30", "HNXINDEX", "HNX30", "UPCOMINDEX"]
cfg.tickers = list(DEFAULT_PRICE_BOARD_TICKERS) if UNIVERSE_MODE == "demo_50" else []
cfg.max_tickers_per_run = len(cfg.tickers) if cfg.tickers else 10_000


def sync_cfg_tickers(cfg: IngestionConfig, *, require_listing: bool = False) -> list[str]:
    if UNIVERSE_MODE == "demo_50":
        cfg.use_listing_as_universe = False
        cfg.tickers = list(DEFAULT_PRICE_BOARD_TICKERS)
        cfg.max_tickers_per_run = len(cfg.tickers)
        return list(cfg.tickers)
    cfg.use_listing_as_universe = True
    if require_listing:
        cfg.tickers = load_tickers_from_listing_bronze(cfg)
        cfg.max_tickers_per_run = len(cfg.tickers)
    elif not cfg.tickers:
        cfg.max_tickers_per_run = 10_000
    return list(cfg.tickers)


def print_config_summary(cfg: IngestionConfig) -> None:
    print(f"profile={NOTEBOOK_PROFILE} universe={UNIVERSE_MODE} board={PRICE_BOARD_MODE}")
    print(f"tickers={len(cfg.tickers)} max_per_run={cfg.max_tickers_per_run}")
    print(f"OHLCV incremental={cfg.use_incremental_window} window={cfg.incremental_window_days}d")
    print(f"price_board listing={cfg.price_board_use_listing_universe} batched={cfg.price_board_batched} cap={cfg.price_board_max_tickers}")
    print(f"run_partition={cfg.run_partition}")


sync_cfg_tickers(cfg)
print_config_summary(cfg)
if UNIVERSE_MODE == "full_listing" and not cfg.tickers:
    print("→ Chạy (1) listing rồi (2) sync_cfg_tickers(cfg, require_listing=True)")


📦 **Vnstock 4.0.4 is available**
Current: 4.0.3 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! (✓ API key saved successfully! vnst***6437
✓ Bạn đang sử dụng gói Cộng đồng (✓ You are using Community Edition - 60 requests/min)
profile=backfill universe=full_listing board=watchlist_50
tickers=0 max_per_run=10000
OHLCV incremental=False window=10d
price_board listing=False batched=False cap=50
run_partition=2026-06-07T154609
→ Chạy (1) listing rồi (2) sync_cfg_tickers(cfg, require_listing=True)


### Pipeline một lệnh

`RUN_ONE_SHOT_PIPELINE=True` → bỏ qua các ô manual.

In [4]:
if not RUN_ONE_SHOT_PIPELINE:
    print("RUN_ONE_SHOT_PIPELINE=False — dùng các ô manual bên dưới.")
else:
    sync_cfg_tickers(cfg, require_listing=(UNIVERSE_MODE == "full_listing"))
    fn = run_structure_full_ingestion_pipeline if INCLUDE_FINANCIAL_RATIO else run_structure_ingestion_pipeline
    fn(cfg)


2026-06-07 15:45:39 [INFO] Loaded listing tickers from D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet: rows_before=3243 rows_after=703 filters=["exchange=['HNX', 'HOSE']", "security_type=['STOCK']"]


2026-06-07 15:45:41 [INFO] symbols_by_exchange: 3243 rows from KBS
2026-06-07 15:45:41 [INFO] Saved raw-ish listing snapshot: 3243 rows -> D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet
2026-06-07 15:45:41 [INFO] Chờ 30s trước bước tiếp theo...


: 

: 

---

## Manual từng bước

Thứ tự `full_listing`: 1→2→3→4→5→6→7.

### (1) Listing


In [ ]:
listing_file = ingest_listing(cfg)
listing_file

### (2) Sync universe

`full_listing`: chạy sau listing.

In [6]:
sync_cfg_tickers(cfg, require_listing=True)
print_config_summary(cfg)

2026-06-04 11:31:41 [INFO] Loaded listing tickers from D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet: rows_before=3244 rows_after=703 filters=["exchange=['HNX', 'HOSE']", "security_type=['STOCK']"]


profile=backfill universe=full_listing board=full_listing
tickers=703 max_per_run=703
OHLCV incremental=False window=10d
price_board listing=True batched=True cap=5000
run_partition=2026-06-04T113101


### (3) Company

In [ ]:
company_file = ingest_company_overview(cfg)
company_file

### (4) Price OHLCV (batch)

In [ ]:
price_result = _ingest_price_batched(cfg)
price_result["batches_ok"], len(price_result["outputs"])

### (5) Index

In [ ]:
ingest_indices(cfg)

### (6) Financial ratio

In [ ]:
ratio = ingest_financial_ratio(cfg)
len(ratio)

### (7) Price board

In [7]:
board = _ingest_price_board_snapshot(cfg)
board["tickers_requested"], board["snapshot_at"], board["outputs"]

2026-06-04 11:31:47 [INFO] Price board fetch batch 1/15: 50 tickers [AAA...BKG] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:31:48 [WARNING] price_board (kbs): No charting library available. Please install one of:
1. vnstock_chart (recommended): pip install --extra-index-url https://vnstocks.com/api/simple vnstock_chart
2. vnstock_ezchart (fallback): pip install vnstock_ezchart


2026-06-04 11:31:55 [INFO] Price board fetch batch 2/15: 50 tickers [BMC...COM] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:00 [INFO] Price board fetch batch 3/15: 50 tickers [CPC...DPG] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:06 [INFO] Price board fetch batch 4/15: 50 tickers [DPM...GHC] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:11 [INFO] Price board fetch batch 5/15: 50 tickers [GIC...HRC] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:16 [INFO] Price board fetch batch 6/15: 50 tickers [HSG...KTS] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:22 [INFO] Price board fetch batch 7/15: 50 tickers [L10...NAP] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:27 [INFO] Price board fetch batch 8/15: 50 tickers [NAV...PGC] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:32 [INFO] Price board fetch batch 9/15: 50 tickers [PGD...PVD] (merge -> 2026-06-04T04-31-47)
2026-06-04 11:32:38 [INFO] Price board fetch batch 10/15: 50 tickers [PVG...SHB] (merge -> 2026-06-04T04-31-47)


(703,
 '2026-06-04T04-31-47',
 ['D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price_board\\snapshot_at=2026-06-04T04-31-47\\PRICE_BOARD_SNAPSHOT.parquet'])

---

## (8) Kiểm tra dữ liệu Bronze đã crawl

Quét `data-lake/raw/Structure_Data/` và tổng hợp số liệu từng dataset: số dòng/ticker, partition mới nhất, số file. Dùng để xác nhận backfill thành công trước khi chạy Silver.

In [3]:
# ====== KIỂM TRA DỮ LIỆU BRONZE ĐÃ CRAWL ======
import pandas as pd
from pathlib import Path

root = cfg.data_lake_root
print(f"Bronze root: {root}\n")


def _latest(glob_parent: Path, pattern: str) -> Path | None:
    if not glob_parent.exists():
        return None
    items = sorted(glob_parent.glob(pattern))
    return items[-1] if items else None


def _safe_rows(p: Path | None) -> int:
    try:
        return len(pd.read_parquet(p)) if p and p.exists() else 0
    except Exception as ex:  # noqa: BLE001
        print(f"  ! đọc lỗi {p}: {ex}")
        return 0


rows = []

# 1) listing
lp = listing_bronze_path(cfg)
if lp.exists():
    ldf = pd.read_parquet(lp)
    rows.append(("listing", lp.parent.name, len(ldf), f"{ldf.shape[1]} cột"))
else:
    rows.append(("listing", "—", 0, "THIẾU"))

# 2) company — snapshot mới nhất
comp = _latest(root / "company" / "snapshots", "snapshot_date=*")
n_comp_snap = len(list((root / "company" / "snapshots").glob("snapshot_date=*"))) if (root / "company" / "snapshots").exists() else 0
rows.append(("company", comp.name if comp else "—", _safe_rows(comp / "company_overview.parquet" if comp else None), f"{n_comp_snap} snapshot"))

# 3) price — OHLCV theo year/month/ticker
price_dir = root / "price"
price_files = [f for f in price_dir.rglob("*.parquet")] if price_dir.exists() else []
price_tickers = sorted({f.stem for f in price_files})
years = sorted({p.name for p in price_dir.glob("year=*")}) if price_dir.exists() else []
rows.append(("price", " ".join(years) or "—", len(price_tickers), f"{len(price_files)} file"))

# 4) index
idx_dir = root / "index"
idx_files = [f for f in idx_dir.rglob("*.parquet")] if idx_dir.exists() else []
rows.append(("index", "", len({f.stem for f in idx_files}), f"{len(idx_files)} file"))

# 5) financial_ratio — snapshot mới nhất
fr = _latest(root / "financial_ratio", "snapshot_date=*")
n_fr_snap = len(list((root / "financial_ratio").glob("snapshot_date=*"))) if (root / "financial_ratio").exists() else 0
n_fr_tickers = len(list(fr.glob("*.parquet"))) if fr else 0
rows.append(("financial_ratio", fr.name if fr else "—", n_fr_tickers, f"{n_fr_snap} snapshot"))

# 6) price_board — snapshot mới nhất
pb = _latest(root / "price_board", "snapshot_at=*")
pb_file = next(pb.glob("*.parquet"), None) if pb else None
rows.append(("price_board", pb.name if pb else "—", _safe_rows(pb_file), f"{len(list((root / 'price_board').glob('snapshot_at=*'))) if (root / 'price_board').exists() else 0} snapshot"))

summary = pd.DataFrame(rows, columns=["dataset", "partition_mới_nhất", "rows/tickers", "ghi_chú"])
print(summary.to_string(index=False))

# Khoảng thời gian giá của 1 mã mẫu (nếu có)
if price_tickers:
    sample = "VCB" if "VCB" in price_tickers else price_tickers[0]
    sdf = pd.concat(
        [pd.read_parquet(f) for f in price_files if f.stem == sample],
        ignore_index=True,
    )
    date_col = next((c for c in ("time", "trading_date", "date") if c in sdf.columns), None)
    if date_col:
        s = pd.to_datetime(sdf[date_col])
        print(f"\nGiá mẫu {sample}: {len(sdf)} dòng, {s.min().date()} → {s.max().date()}")
    print(f"Cột price: {list(sdf.columns)}")

Bronze root: D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data

        dataset                                          partition_mới_nhất  rows/tickers    ghi_chú
        listing                                                      master          3243      8 cột
        company                                    snapshot_date=2026-06-07            42 2 snapshot
          price year=2021 year=2022 year=2023 year=2024 year=2025 year=2026           985 42013 file
          index                                                                         5   305 file
financial_ratio                                    snapshot_date=2026-06-07            20 2 snapshot
    price_board                             snapshot_at=2026-06-04T04-31-47           703 3 snapshot

Giá mẫu VCB: 1249 dòng, 2021-06-03 → 2026-06-05
Cột price: ['time', 'open', 'high', 'low', 'close', 'volume', 'ticker', 'ingested_at', 'fetched_at', 'source', 'instrument_type', 'trading_date']
